# SemEval 2022 Task 2A: Multilingual Idiomaticity Detection
- **Objective**: Binary classification of Multi-Word Expressions (MWEs) as idiomatic (0) or literal (1)
- **Languages**: English (EN) and Portuguese (PT)
- **Model**: bert-base-multilingual-cased (mBERT)
- All hyperparameters match official implementation

### 1. Setup and Installation


In [1]:
# Install dependencies
!pip install -q torch transformers pandas scikit-learn sentencepiece datasets

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import torch
from pathlib import Path
import os
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed
)
from datasets import Dataset

# Set seed for reproducibility
SEED = 0
set_seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.9.0+cu126
CUDA available: True


In [11]:
!unzip /content/SubTaskA.zip

Archive:  /content/SubTaskA.zip
replace SubTaskA/SubTaskA/Data/dev.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: SubTaskA/SubTaskA/Data/dev.csv  
  inflating: SubTaskA/SubTaskA/Data/dev_gold.csv  
  inflating: SubTaskA/SubTaskA/Data/dev_submission_format.csv  
  inflating: SubTaskA/SubTaskA/Data/eval.csv  
  inflating: SubTaskA/SubTaskA/Data/eval_submission_format.csv  
  inflating: SubTaskA/SubTaskA/Data/train_one_shot.csv  
  inflating: SubTaskA/SubTaskA/Data/train_zero_shot.csv  
  inflating: SubTaskA/SubTaskA/TestData/test.csv  
  inflating: SubTaskA/SubTaskA/TestData/test_submission_format.csv  
  inflating: SubTaskA/SubTaskA/TestData/train_one_shot.csv  


### 2. Data Loading and Pre-processing

#### Ablation study (with and without context)
- FULL CONTEXT (Baseline): Previous + Target + Next
- TARGET ONLY: Just the MWE
- TARGET + PREVIOUS: Target with left context
- TARGET + NEXT: Target with right context

In [30]:
# Choose context variant
CONTEXT_MODE = "TARGET_NEXT"

# Colab paths
INPUT_DIR = Path("/content/SubTaskA/SubTaskA/Data/")
OUTPUT_DIR = Path(f"models/ZeroShot/{CONTEXT_MODE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"CONTEXT MODE: {CONTEXT_MODE}")


CONTEXT MODE: TARGET_NEXT


In [31]:
# Defining functions to build input sentences based on context mode
# Each function takes a row of the DataFrame and constructs the input sentence
def build_sentence_full(row):
    """BASELINE: Previous + Target + Next """
    prev = str(row['Previous']).strip() if pd.notna(row['Previous']) else ""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    nxt = str(row['Next']).strip() if pd.notna(row['Next']) else ""
    return f"{prev} {target} {nxt}".strip()

def build_sentence_target_only(row):
    """ABLATION 1: Target only (no context)"""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    return target

def build_sentence_target_prev(row):
    """ABLATION 2: Target + Previous (left context only)"""
    prev = str(row['Previous']).strip() if pd.notna(row['Previous']) else ""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    return f"{prev} {target}".strip()

def build_sentence_target_next(row):
    """ABLATION 3: Target + Next (right context only)"""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    nxt = str(row['Next']).strip() if pd.notna(row['Next']) else ""
    return f"{target} {nxt}".strip()

# Map context mode to function
CONTEXT_FUNCTIONS = {
    "FULL": build_sentence_full,
    "TARGET_ONLY": build_sentence_target_only,
    "TARGET_PREV": build_sentence_target_prev,
    "TARGET_NEXT": build_sentence_target_next
}

build_sentence = CONTEXT_FUNCTIONS[CONTEXT_MODE]

In [32]:
# Load and Preprocess Data
print("\nLoading datasets")

# Train Data
train_df = pd.read_csv(INPUT_DIR / "train_zero_shot.csv")
train_data = pd.DataFrame({
    "label": train_df["Label"],
    "text": train_df.apply(build_sentence, axis=1)
})
print(f" Train: {len(train_data)} samples")

# Dev Data (merge with gold labels)
dev_df = pd.read_csv(INPUT_DIR / "dev.csv")
dev_gold = pd.read_csv(INPUT_DIR / "dev_gold.csv")
dev_df = dev_df.merge(dev_gold[["ID", "Label"]], on="ID", how="left")
dev_data = pd.DataFrame({
    "label": dev_df["Label"],
    "text": dev_df.apply(build_sentence, axis=1)
})
print(f" Dev: {len(dev_data)} samples")

# Distribution analysis
print("\nDataset Statistics")
print(f"Train - Label 0 (Idiomatic): {(train_data['label']==0).sum()} ({(train_data['label']==0).sum()/len(train_data)*100:.1f}%)")
print(f"Train - Label 1 (Literal): {(train_data['label']==1).sum()} ({(train_data['label']==1).sum()/len(train_data)*100:.1f}%)")
print(f"Dev   - Label 0 (Idiomatic): {(dev_data['label']==0).sum()} ({(dev_data['label']==0).sum()/len(dev_data)*100:.1f}%)")
print(f"Dev   - Label 1 (Literal): {(dev_data['label']==1).sum()} ({(dev_data['label']==1).sum()/len(dev_data)*100:.1f}%)")


Loading datasets
 Train: 4491 samples
 Dev: 739 samples

Dataset Statistics
Train - Label 0 (Idiomatic): 2535 (56.4%)
Train - Label 1 (Literal): 1956 (43.6%)
Dev   - Label 0 (Idiomatic): 336 (45.5%)
Dev   - Label 1 (Literal): 403 (54.5%)


### 3. Model Parameters

#### Fixed hyperparameters as used in Baseline
- Model: `bert-base-multilingual-cased`
- Learning rate: `2e-5`
- Batch size: `32`
- Epochs: `9`
- Max sequence length: `128`
- Seed: `0`

In [33]:
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128

print(f"Loading model and tokenizer: {MODEL_NAME}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model (num_labels=2 for binary classification)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
# Note: Warning about "classifier weights not initialized" is EXPECTED
# The warning is because we are loading a pre-trained model without a classification head
# The classification head is randomly initialized and will be trained

Loading model and tokenizer: bert-base-multilingual-cased


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 4. Tokenisation

In [34]:
# Tokenisation: it takes raw text and converts it into input IDs and attention masks.
def tokenize_function(examples):
    """Tokenize text with padding and truncation"""
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

print("Tokenizing datasets")

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_data)
dev_dataset = Dataset.from_pandas(dev_data)

# Apply tokenization to all samples
train_dataset = train_dataset.map(tokenize_function, batched=True)
dev_dataset = dev_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch tensors so that Trainer can use them
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
dev_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f" Train dataset tokenized: {len(train_dataset)} samples")
print(f" Dev dataset tokenized: {len(dev_dataset)} samples")

Tokenizing datasets


Map:   0%|          | 0/4491 [00:00<?, ? examples/s]

Map:   0%|          | 0/739 [00:00<?, ? examples/s]

 Train dataset tokenized: 4491 samples
 Dev dataset tokenized: 739 samples


### 4. TRAINING

Training with fixed hyperparameters matching the official baseline.
- Early stopping enabled based on dev set F1 score
- Best checkpoint saved automatically

In [35]:
# Metric computation function
def compute_metrics(eval_pred):
    """Calculate Macro F1 score (average of F1 for each class)"""
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    macro_f1 = f1_score(labels, predictions, average="macro")
    return {"macro-f1": macro_f1}

# Training arguments
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # Hyperparameters (Baseline replication hence fixed)
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=9,

    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro-f1",

    # Reproducibility
    seed=SEED,

    # Logging
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=50,
    report_to="none"
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Training configuration:\n")
print(f"Model: {MODEL_NAME}")
print(f"Context mode: {CONTEXT_MODE}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Max length: {MAX_LENGTH}")
print(f"Seed: {SEED}")
print(f"Output dir: {OUTPUT_DIR}")


/tmp/ipython-input-994343218.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training configuration:

Model: bert-base-multilingual-cased
Context mode: TARGET_NEXT
Learning rate: 2e-05
Batch size: 32
Epochs: 9
Max length: 128
Seed: 0
Output dir: models/ZeroShot/TARGET_NEXT


In [36]:
print("Model Training in Progress\n")

# Train model
train_result = trainer.train()

print("\n✅ Training complete!")
print(f"Best model checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best Macro-F1 score: {trainer.state.best_metric:.4f}")

Model Training in Progress



Epoch,Training Loss,Validation Loss,Macro-f1
1,0.560100,0.692229,0.662702
2,0.219900,1.058117,0.660099
3,0.092800,1.295343,0.679880
4,0.058900,1.867274,0.656496
5,0.027800,1.905109,0.674854
6,0.028000,2.122395,0.681405
7,0.016000,2.262995,0.667078
8,0.000500,2.336878,0.676278
9,0.001300,2.351159,0.669816



✅ Training complete!
Best model checkpoint: models/ZeroShot/TARGET_NEXT/checkpoint-846
Best Macro-F1 score: 0.6814


### 5. Evaluation
Analysing results with respect to languages and classes in dataset
- Class (Idiomatic vs Literal)
- Language (EN vs PT)
- Combined metrics

In [37]:
print("Generating predictions on dev set")

# Get predictions
predictions = trainer.predict(dev_dataset)
pred_labels = predictions.predictions.argmax(axis=-1)

# Load language information for detailed analysis
dev_full = pd.read_csv(INPUT_DIR / "dev.csv")
dev_full = dev_full.merge(dev_gold[["ID", "Label"]], on="ID", how="left")

# Create results dataframe
results_df = pd.DataFrame({
    'ID': dev_full['ID'],
    'Language': dev_full['Language'],
    'Gold_Label': dev_full['Label'],
    'Predicted_Label': pred_labels
})

print(f"Predictions generated: {len(results_df)} samples")

Generating predictions on dev set


Predictions generated: 739 samples


In [38]:
print(f"Evaluation results for context mode: {CONTEXT_MODE}")

print(f"\nTotal samples: {len(results_df)}")
print(f"Languages: {sorted(results_df['Language'].unique())}")
print(f"Classes: 0 (Idiomatic), 1 (Literal)")
print("-"*80)

# Results for combinened classes
print("\nOverall Results")
print("\n" + classification_report(
    results_df['Gold_Label'],
    results_df['Predicted_Label'],
    target_names=['Idiomatic', 'Literal'],
    digits=4
))

# Macro-F1 for combined classes
macro_f1 = f1_score(results_df['Gold_Label'], results_df['Predicted_Label'], average='macro')


print("-"*80)
print("\nResults by Language")
for lang in sorted(results_df['Language'].unique()):
    lang_data = results_df[results_df['Language'] == lang]

    print(f"\n{lang} ({len(lang_data)} samples)")
    print(classification_report(
        lang_data['Gold_Label'],
        lang_data['Predicted_Label'],
        target_names=['Idiomatic', 'Literal'],
        digits=4
    ))

# Calculate macro F1 for each language
en_data = results_df[results_df['Language'] == 'EN']
pt_data = results_df[results_df['Language'] == 'PT']

en_macro_f1 = f1_score(en_data['Gold_Label'], en_data['Predicted_Label'], average='macro')
pt_macro_f1 = f1_score(pt_data['Gold_Label'], pt_data['Predicted_Label'], average='macro')

print("-"*80)
print(f"\nSummary of Macro F1 Scores for {CONTEXT_MODE}")
summary_data = [
    {'Metric': 'EN Macro F1', 'Value': f"{en_macro_f1:.4f}"},
    {'Metric': 'PT Macro F1', 'Value': f"{pt_macro_f1:.4f}"},
    {'Metric': 'Combined Macro F1', 'Value': f"{macro_f1:.4f}"}
]
summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

Evaluation results for context mode: TARGET_NEXT

Total samples: 739
Languages: ['EN', 'PT']
Classes: 0 (Idiomatic), 1 (Literal)
--------------------------------------------------------------------------------

Overall Results

              precision    recall  f1-score   support

   Idiomatic     0.6361    0.7024    0.6676       336
     Literal     0.7283    0.6650    0.6952       403

    accuracy                         0.6820       739
   macro avg     0.6822    0.6837    0.6814       739
weighted avg     0.6864    0.6820    0.6827       739

--------------------------------------------------------------------------------

Results by Language

EN (466 samples)
              precision    recall  f1-score   support

   Idiomatic     0.6250    0.6044    0.6145       182
     Literal     0.7517    0.7676    0.7596       284

    accuracy                         0.7039       466
   macro avg     0.6884    0.6860    0.6871       466
weighted avg     0.7022    0.7039    0.7029       466

---
## 6. ABLATION STUDIES - CONTEXT VARIATIONS

### Instructions for Running Ablations:

To test different context configurations:

1. **Go back to Cell: "CONFIGURATION: Choose context variant"**
2. **Change `CONTEXT_MODE`** to one of:
   - `"FULL"` - Previous + Target + Next (baseline)
   - `"TARGET_ONLY"` - Just the MWE (no context)
   - `"TARGET_PREV"` - Target + Previous
   - `"TARGET_NEXT"` - Target + Next

3. **Re-run all cells from Section 2 onwards**

4. **Record results in the table below**

### Expected Ablation Results Template:

| Context Mode | EN F1 | PT F1 | Combined F1 | Notes |
|--------------|-------|-------|-------------|-------|
| FULL (Baseline) | ?.???? | ?.???? | ?.???? | Official baseline replication |
| TARGET_ONLY | ?.???? | ?.???? | ?.???? | No context - tests intrinsic idiomaticity |
| TARGET_PREV | ?.???? | ?.???? | ?.???? | Left context only |
| TARGET_NEXT | ?.???? | ?.???? | ?.???? | Right context only |

### Research Questions:
- Does context help idiom detection?
- Is there asymmetry between left and right context?
- Do EN and PT differ in context dependency?

---
## 📝 NOTES

### Reproducibility Checklist:
- ✅ Seed fixed at 0 (matches official baseline)
- ✅ All hyperparameters match official implementation
- ✅ Same model: bert-base-multilingual-cased
- ✅ Same evaluation metric: Macro F1
- ✅ Same data preprocessing: context concatenation

### Expected Performance:
- Official baseline reports ~0.75-0.80 Macro F1 on dev set
- Minor variations (<0.02) acceptable due to hardware/library differences

### For Research Paper:
```
We replicated the official baseline using bert-base-multilingual-cased
with fixed hyperparameters (lr=2e-5, batch=32, epochs=9, seed=0).
The model was trained on concatenated context (Previous + Target + Next)
and achieved [X.XX] Macro F1 on the development set.
```